In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load data
df = pd.read_csv('softec-25-machine-learning-competition\\train.csv', parse_dates=['STAY_FROM_DT', 'STAY_THRU_DT'])

# 1. Handle Missing Values
print("Missing values before cleaning:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

# Fill missing diagnosis/procedure codes with 'NO_CODE'
diagnosis_cols = [f'DGNSCD{i:02d}' for i in range(1, 26)]
procedure_cols = [f'PRCDRCD{i:02d}' for i in range(1, 26)]

for col in diagnosis_cols + procedure_cols:
    df[col] = df[col].fillna('NO_CODE')

# Fill other categoricals with mode
df['STAY_DRG_CD'] = df['STAY_DRG_CD'].fillna('UNKNOWN_DRG')
df['TYPE_ADM'] = df['TYPE_ADM'].fillna(df['TYPE_ADM'].mode()[0])
df['SRC_ADMS'] = df['SRC_ADMS'].fillna(df['SRC_ADMS'].mode()[0])

# 2. Feature Engineering - Stay Duration
df['stay_duration_days'] = (df['STAY_THRU_DT'] - df['STAY_FROM_DT']).dt.days

# 3. Code Frequency Features
df['num_diagnoses'] = df[diagnosis_cols].apply(lambda x: x[x != 'NO_CODE'].count(), axis=1)
df['num_procedures'] = df[procedure_cols].apply(lambda x: x[x != 'NO_CODE'].count(), axis=1)

# 4. Simplify High-Cardinality Codes (extract first 3 characters)
for col in diagnosis_cols:
    df[f'{col}_category'] = df[col].str[:3]
    
for col in procedure_cols:
    df[f'{col}_category'] = df[col].str[:3]

# 5. Drop duplicates and irrelevant columns
df = df.drop(columns=['stay_drg_cd'])  # duplicate column
df = df.drop_duplicates(subset=['ID'])

# 6. Check cleaned data
print("\nMissing values after cleaning:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nSample cleaned data:")
print(df[['ID', 'stay_duration_days', 'num_diagnoses', 'num_procedures']].head())

Missing values before cleaning:
STAY_DRG_CD    126498
PRCDRCD25      125908
PRCDRCD24      125897
PRCDRCD23      125887
PRCDRCD22      125873
PRCDRCD21      125859
PRCDRCD20      125842
PRCDRCD19      125821
PRCDRCD18      125790
PRCDRCD17      125760
PRCDRCD16      125714
PRCDRCD15      125648
PRCDRCD14      125534
PRCDRCD13      125405
PRCDRCD12      125232
PRCDRCD11      124998
PRCDRCD10      124653
PRCDRCD09      124251
PRCDRCD08      123643
PRCDRCD07      122663
dtype: int64


C:\Users\shoai\AppData\Local\Temp\ipykernel_17548\1418893776.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_category'] = df[col].str[:3]
C:\Users\shoai\AppData\Local\Temp\ipykernel_17548\1418893776.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_category'] = df[col].str[:3]
C:\Users\shoai\AppData\Local\Temp\ipykernel_17548\1418893776.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo


Missing values after cleaning:
ID                   0
STAY_DRG_CD          0
DGNSCD21_category    0
DGNSCD20_category    0
DGNSCD19_category    0
DGNSCD18_category    0
DGNSCD17_category    0
DGNSCD16_category    0
DGNSCD15_category    0
DGNSCD14_category    0
dtype: int64

Sample cleaned data:
      ID  stay_duration_days  num_diagnoses  num_procedures
0  17319                   7             25               1
1  19722                   4             15               0
2  89699                   2             21               0
3   8086                   7             22               0
4  68049                   6             13               0


In [10]:
import os


output_file = 'cleaned_train_data.csv'

print("\nFinal Data Summary:")
print(f"- Total records: {len(df)}")
print(f"- Columns: {df.columns.tolist()}")
print(f"- Readmission rate: {df['Readmitted_30'].mean():.2%}")

if df.isnull().sum().sum() > 0:
    print("\nWarning: Missing values detected in these columns:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print("\nNo missing values remaining - safe to export")

df.to_csv(output_file, 
          index=False,          
          encoding='utf-8',     
          date_format='%Y-%m-%d') 

print(f"\nSuccessfully exported to {output_file}")
print(f"File size: {os.path.getsize(output_file)/1024:.2f} KB")


pd.read_csv(output_file, nrows=5).head()


Final Data Summary:
- Total records: 130296
- Columns: ['ID', 'STAY_DRG_CD', 'STAY_FROM_DT', 'STAY_THRU_DT', 'STUS_CD', 'TYPE_ADM', 'SRC_ADMS', 'AD_DGNS', 'DGNSCD01', 'PRCDRCD01', 'DGNSCD02', 'PRCDRCD02', 'DGNSCD03', 'PRCDRCD03', 'DGNSCD04', 'PRCDRCD04', 'DGNSCD05', 'PRCDRCD05', 'DGNSCD06', 'PRCDRCD06', 'DGNSCD07', 'PRCDRCD07', 'DGNSCD08', 'PRCDRCD08', 'DGNSCD09', 'PRCDRCD09', 'DGNSCD10', 'PRCDRCD10', 'DGNSCD11', 'PRCDRCD11', 'DGNSCD12', 'PRCDRCD12', 'DGNSCD13', 'PRCDRCD13', 'DGNSCD14', 'PRCDRCD14', 'DGNSCD15', 'PRCDRCD15', 'DGNSCD16', 'PRCDRCD16', 'DGNSCD17', 'PRCDRCD17', 'DGNSCD18', 'PRCDRCD18', 'DGNSCD19', 'PRCDRCD19', 'DGNSCD20', 'PRCDRCD20', 'DGNSCD21', 'PRCDRCD21', 'DGNSCD22', 'PRCDRCD22', 'DGNSCD23', 'PRCDRCD23', 'DGNSCD24', 'PRCDRCD24', 'DGNSCD25', 'PRCDRCD25', 'Readmitted_30', 'stay_duration_days', 'num_diagnoses', 'num_procedures', 'DGNSCD01_category', 'DGNSCD02_category', 'DGNSCD03_category', 'DGNSCD04_category', 'DGNSCD05_category', 'DGNSCD06_category', 'DGNSCD07_category'

,ID,STAY_DRG_CD,STAY_FROM_DT,STAY_THRU_DT,STUS_CD,TYPE_ADM,SRC_ADMS,AD_DGNS,DGNSCD01,PRCDRCD01,...,PRCDRCD16_category,PRCDRCD17_category,PRCDRCD18_category,PRCDRCD19_category,PRCDRCD20_category,PRCDRCD21_category,PRCDRCD22_category,PRCDRCD23_category,PRCDRCD24_category,PRCDRCD25_category
0,17319,UNKNOWN_DRG,2017-12-13,2017-12-20,62,1,2,M25551,S72001A,0SRR01Z,...,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_
1,19722,UNKNOWN_DRG,2017-10-19,2017-10-23,1,1,1,R531,A419,NO_CODE,...,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_
2,89699,UNKNOWN_DRG,2018-08-06,2018-08-08,1,1,1,R002,J690,NO_CODE,...,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_
3,8086,UNKNOWN_DRG,2016-12-20,2016-12-27,62,5,1,K661,K661,NO_CODE,...,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_
4,68049,UNKNOWN_DRG,2016-01-06,2016-01-12,6,1,1,J9601,J690,NO_CODE,...,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_,NO_


<bound method NDFrame.first of             ID  STAY_DRG_CD STAY_FROM_DT STAY_THRU_DT  STUS_CD  TYPE_ADM  \
0        17319  UNKNOWN_DRG   2017-12-13   2017-12-20       62         1   
1        19722  UNKNOWN_DRG   2017-10-19   2017-10-23        1         1   
2        89699  UNKNOWN_DRG   2018-08-06   2018-08-08        1         1   
3         8086  UNKNOWN_DRG   2016-12-20   2016-12-27       62         5   
4        68049  UNKNOWN_DRG   2016-01-06   2016-01-12        6         1   
...        ...          ...          ...          ...      ...       ...   
130291  119880  UNKNOWN_DRG   2018-12-18   2018-12-19        1         3   
130292  103695  UNKNOWN_DRG   2019-11-23   2019-11-27        6         1   
130293  131933  UNKNOWN_DRG   2020-12-10   2020-12-17        6         1   
130294  146868  UNKNOWN_DRG   2022-10-10   2022-10-19        3         1   
130295  121959  UNKNOWN_DRG   2019-04-30   2019-05-03        3         1   

        SRC_ADMS AD_DGNS DGNSCD01 PRCDRCD01  ... PRCDRCD